# Entrenamiento de Modelo para Clasificación - Dataset Iris

Entrenamos un modelo lineal con regularización **L1 (tipo lasso)** para predecir la **especie de la planta** (`setosa`, `versicolor`, `virginica`) a partir de sus características.

In [21]:
# Importar librerías
from datetime import datetime, timezone
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

In [22]:
# Cargar dataset Iris
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [23]:
# Separar features y target
# Ahora predecimos la clase (tipo de iris)
X = df[["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]]
y = iris.target
class_names = iris.target_names

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (120, 4) | Test: (30, 4)


In [26]:
# Entrenar pipeline: escalado + modelo de clasificación con regularización L1 (tipo lasso)
pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            solver="saga",
            l1_ratio=1.0,
            max_iter=2000,
            random_state=42
        ))
    ]
)
pipeline.fit(X_train, y_train)

# Evaluar
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print("\nMatriz de confusión:")
print(cm)
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=class_names))

Accuracy: 0.9333

Matriz de confusión:
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]

Reporte de clasificación:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



In [25]:
# Guardar pipeline en carpeta models con metadata del modelo
from pathlib import Path

models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

# Metadata automática para que la app pueda leerla desde el propio pipeline
pipeline.model_name = "iris_logreg_l1_pipeline"
pipeline.model_version = datetime.now(timezone.utc).strftime("v%Y%m%d%H%M%S")
pipeline.feature_names = list(X.columns)

joblib.dump(pipeline, models_dir / "pipeline.joblib")
print(
    f"Pipeline guardado: models/pipeline.joblib | "
    f"model_name={pipeline.model_name} | model_version={pipeline.model_version}"
)

Pipeline guardado: models/pipeline.joblib | model_name=iris_logreg_l1_pipeline | model_version=v20260405210532
